# Pretty Formula One

This notebook is used to fetch data from certain seasons and save them into files to be uploaded into hosted database

In [1]:
YEAR = 2026

## Drivers

In [2]:
import fastf1
import json
from pprint import pprint
import os
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)

session = fastf1.get_session(YEAR, 'Monaco', 'R')
session.load(laps=False, telemetry=False, weather=False)
drivers_list = []
seen_drivers = set()
for _, row in session.results.iterrows():
    driver_slug = f"{row['DriverId']}_{YEAR}"
    if driver_slug not in seen_drivers:
        drivers_list.append({
            "id": driver_slug,
            "team": row["TeamName"],
            "abbreviation": row["Abbreviation"],
            "name": row["FullName"],
            "teamLogo": f"/assets/icons/{row['TeamName'].lower().replace(' ', '-')}.png",
        })
        seen_drivers.add(driver_slug)
        
os.makedirs(f'data/{YEAR}', exist_ok=True)
with open(f"./data/{YEAR}/drivers_{YEAR}.json", "w") as f:
    json.dump(drivers_list, f, separators=(',', ':'))

## Race Results

In [11]:
import fastf1
import json
import os
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)

os.makedirs(f'data/{YEAR}', exist_ok=True)
with open(f"./data/{YEAR}/drivers_{YEAR}.json", "r") as f:
    drivers_list = json.load(f)
    
schedule = fastf1.get_event_schedule(YEAR)
events = schedule["EventName"].to_list()
events = [event for event in events if "Testing" not in event]
races = [idx+1 for idx, _ in enumerate(events)]
    
all_rounds_data = []
for ROUND_ID in races:

    race_session = fastf1.get_session(YEAR, ROUND_ID, "R")
    race_session.load(telemetry=False, weather=False)

    race_points_map = {
        f"{row['DriverId']}_{YEAR}": int(row['Points']) 
        for _, row in race_session.results.iterrows()
    }

    id_to_number_map = {
        f"{row['DriverId']}_{YEAR}": row['DriverNumber'] 
        for _, row in race_session.results.iterrows()
    }

    sprint_points_map = {}
    has_sprint = False
    try:
        sprint_session = fastf1.get_session(YEAR, ROUND_ID, "S")
        sprint_session.load(laps=False, telemetry=False, weather=False)
        has_sprint = True
        sprint_points_map = {
            row['DriverNumber']: int(row['Points']) 
            for _, row in sprint_session.results.iterrows()
        }
    except Exception:
        pass

    race_results_list = []
    for driver in drivers_list:
        d_id = driver["id"]
        r_points = race_points_map.get(d_id, 0)
        s_points = 0
        if has_sprint:
            d_number = id_to_number_map.get(d_id)
            if d_number:
                s_points = sprint_points_map.get(d_number, 0)
                
        laps = race_session.laps
        tyre_data = laps[['Compound', 'LapNumber', 'DriverNumber', 'Stint']]
        lap_tyre_data = {}
        for _, row in tyre_data.iterrows():
            if row["Compound"] == "nan":
                continue
            driver_number = row["DriverNumber"]
            if driver_number not in lap_tyre_data:
                lap_tyre_data[driver_number] = []
            last_compound = lap_tyre_data[driver_number][-1]["compound"] if lap_tyre_data[driver_number] else None
            if row["Compound"] != last_compound:
                lap_tyre_data[driver_number].append({
                    "compound": row["Compound"],
                    "lapStart": row["LapNumber"],
                    "lapEnd": row["LapNumber"],
                    "stint": row["Stint"]
                })
            else:
                lap_tyre_data[driver_number][-1]["lapEnd"] = row["LapNumber"]

        race_results_list.append({
            "driver_id": d_id,
            "racePoints": r_points,
            "sprintPoints": s_points,
            "tyre_strat": lap_tyre_data.get(id_to_number_map.get(d_id), [])
        })

    event_info = race_session.event
    country = event_info["Country"]
    schedule = fastf1.get_event_schedule(YEAR)
    total_rounds = len(schedule[schedule["RoundNumber"] > 0])

    round_data = {
        "id": int(event_info["RoundNumber"]),
        "year": YEAR,
        "index": int(event_info["RoundNumber"]),
        "totalRounds": total_rounds,
        "name": event_info["EventName"],
        "nameVerbose": event_info["OfficialEventName"],
        "country": country,
        "backgroundImage": f"/assets/circuits/{country.lower().replace(' ', '_')}.png",
        "results": sorted(race_results_list, key=lambda x: x["racePoints"], reverse=True),
    }
    
    print(f"Processed round {ROUND_ID}: {event_info['EventName']}")

    all_rounds_data.append(round_data)
    
filename = f"./data/{YEAR}/rounds_{YEAR}.json"

os.makedirs(f'data/{YEAR}', exist_ok=True)
with open(filename, "w") as f:
    json.dump(all_rounds_data, f, separators=(',', ':'))

Processed round 1: Bahrain Grand Prix
Processed round 2: Saudi Arabian Grand Prix
Processed round 3: Australian Grand Prix
Processed round 4: Azerbaijan Grand Prix
Processed round 5: Miami Grand Prix
Processed round 6: Monaco Grand Prix
Processed round 7: Spanish Grand Prix
Processed round 8: Canadian Grand Prix
Processed round 9: Austrian Grand Prix
Processed round 10: British Grand Prix
Processed round 11: Hungarian Grand Prix
Processed round 12: Belgian Grand Prix
Processed round 13: Dutch Grand Prix
Processed round 14: Italian Grand Prix
Processed round 15: Singapore Grand Prix
Processed round 16: Japanese Grand Prix
Processed round 17: Qatar Grand Prix
Processed round 18: United States Grand Prix
Processed round 19: Mexico City Grand Prix
Processed round 20: São Paulo Grand Prix
Processed round 21: Las Vegas Grand Prix
Processed round 22: Abu Dhabi Grand Prix


## All Event Names and Weekend Dates

In [12]:
# getting all the name of the grand prix on the season
import fastf1
import json
from pprint import pprint
import os
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)

schedule = fastf1.get_event_schedule(YEAR)
events_dates = []
for event_name in schedule["EventName"].to_list():
    date = schedule[schedule["EventName"] == event_name]["Session1Date"].values[0]
    date = date.strftime("%Y-%m-%dT%H:%M:%S%z")
    events_dates.append({
        "name": event_name.replace("Grand Prix", "").strip(),
        "date": date,
        "season": YEAR,
    })
pprint(events_dates)
os.makedirs(f'data/{YEAR}', exist_ok=True)
with open(f"./data/{YEAR}/dates_{YEAR}.json", "w") as f:
    json.dump(events_dates, f, separators=(',', ':'))

[{'date': '2023-02-23T10:00:00+0300',
  'name': 'Pre-Season Testing',
  'season': 2023},
 {'date': '2023-03-03T14:30:00+0300', 'name': 'Bahrain', 'season': 2023},
 {'date': '2023-03-17T16:30:00+0300', 'name': 'Saudi Arabian', 'season': 2023},
 {'date': '2023-03-31T12:30:00+1000', 'name': 'Australian', 'season': 2023},
 {'date': '2023-04-28T13:30:00+0400', 'name': 'Azerbaijan', 'season': 2023},
 {'date': '2023-05-05T14:00:00-0400', 'name': 'Miami', 'season': 2023},
 {'date': '2023-05-26T13:30:00+0200', 'name': 'Monaco', 'season': 2023},
 {'date': '2023-06-02T13:30:00+0200', 'name': 'Spanish', 'season': 2023},
 {'date': '2023-06-16T13:30:00-0400', 'name': 'Canadian', 'season': 2023},
 {'date': '2023-06-30T13:30:00+0200', 'name': 'Austrian', 'season': 2023},
 {'date': '2023-07-07T12:30:00+0100', 'name': 'British', 'season': 2023},
 {'date': '2023-07-21T13:30:00+0200', 'name': 'Hungarian', 'season': 2023},
 {'date': '2023-07-28T13:30:00+0200', 'name': 'Belgian', 'season': 2023},
 {'date': 